# 04 — SARIMAX

Estimation, diagnostics, and the conditional against operational covariate
comparison.

**Report sections fed:** 6 (SARIMAX).


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


## Exogenous covariates\n\nOutdoor weather only. Indoor sensors are excluded here — see report Section 4.3.

In [ ]:
weather = [c for c in config.WEATHER_COLS if c in frame.columns]
exog = frame[weather]
print(weather)


## Estimation\n\nOrders follow from the stationarity tests in notebook 02.

In [ ]:
fit = sarimax.fit_sarimax(y_train, exog_train=exog.loc[y_train.index])
print(f"AIC {fit.aic:.1f}   BIC {fit.bic:.1f}")
fit.summary()


### Built-in residual diagnostics

In [ ]:
fig = fit.plot_diagnostics(figsize=(13, 9))
fig.tight_layout()


## Conditional against operational

`conditional` uses realised test-set weather; `operational` persists the last
pre-origin observation. The gap is the cost of covariate uncertainty.


In [ ]:
forecasts = {
    "sarimax_conditional": sarimax.rolling_origin_sarimax(
        fit, y, test_index, config.HORIZON, exog=exog),
    "sarimax_operational": sarimax.rolling_origin_sarimax(
        fit, y, test_index, config.HORIZON,
        exog=sarimax.persisted_exog(exog, test_index, config.HORIZON)),
}

evaluation.evaluate_all(forecasts, y_test, y_train).round(3)


### Target-only variant\n\nDoes weather contribute anything at all?

In [ ]:
fit_plain = sarimax.fit_sarimax(y_train)
forecasts["sarimax_target_only"] = sarimax.rolling_origin_sarimax(
    fit_plain, y, test_index, config.HORIZON)

evaluation.evaluate_all(forecasts, y_test, y_train).round(3)
